In [ ]:
import torch
from torch_geometric.loader import DataLoader
from dataclasses import dataclass
from pathlib import Path

In [ ]:


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    base_model_path: Path
    training_data_path: Path
    test_data_path: Path
    params_epochs: int
    params_batch_size: int
    params_learning_rate: float



In [ ]:
from src.GNNClassfier.constants import *
from src.GNNClassfier.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        # read_yaml returns a ConfigBox for attribute-style access
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_training_config(self) -> TrainingConfig:
        training_config = TrainingConfig(
            root_dir = self.config.artifacts_root,
            trained_model_path = self.config.training.trained_model_path,
            base_model_path = self.config.prepare_base_model.base_model_path,
            training_data_path = self.config.training.training_data_path,
            test_data_path = self.config.training.test_data_path,
            params_epochs = self.params.EPOCHS,
            params_batch_size = self.params.BATCH_SIZE,
            params_learning_rate = self.params.LEARNING_RATE
        )
        return training_config

In [ ]:
class ModelTrainer:
    def __init__(self, config: TrainingConfig):
        self.config = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def train_model(self):
        # 1. Helper function to load all .pt files from the processed directory
        def load_graphs_from_dir(directory_path):
            processed_dir = Path(directory_path) / "processed"
            
            if not processed_dir.exists():
                raise FileNotFoundError(f"The directory {processed_dir} does not exist. Check your Stage 03 output.")
            
            # Find all .pt files and sort them to maintain consistency
            graph_files = sorted([str(f) for f in processed_dir.glob("*.pt")])
            
            if len(graph_files) == 0:
                raise ValueError(f"No .pt files found in {processed_dir}")
                
            print(f"Loading {len(graph_files)} graphs from {processed_dir}...")
            return [torch.load(f, weights_only = False) for f in graph_files]

        # 2. Load Data
        train_data = load_graphs_from_dir(self.config.training_data_path)
        test_data = load_graphs_from_dir(self.config.test_data_path)
        
        train_loader = DataLoader(train_data, batch_size=self.config.params_batch_size, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=self.config.params_batch_size, shuffle=False)

        # 3. Load Model
        # Assuming the base model was saved as a whole object in stage 04
        model = torch.load(self.config.base_model_path, weights_only = False).to(self.device)
        optimizer = torch.optim.Adam(model.parameters(), lr=self.config.params_learning_rate)
        criterion = torch.nn.CrossEntropyLoss()

        # 4. Training Loop
        print(f"Starting training on {self.device} for {self.config.params_epochs} epochs...")
        for epoch in range(1, self.config.params_epochs + 1):
            model.train()
            total_loss = 0
            for data in train_loader:
                data = data.to(self.device)
                optimizer.zero_grad()
                
                # Forward pass
                out = model(data.x.float(), data.edge_index, data.batch)
                
                # Compute loss (assuming y is the label)
                loss = criterion(out, data.y.long().squeeze())
                
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            
            # Validation
            model.eval()
            correct = 0
            with torch.no_grad():
                for data in test_loader:
                    data = data.to(self.device)
                    out = model(data.x.float(), data.edge_index, data.batch)
                    pred = out.argmax(dim=1)
                    
                    # Handle potential dimension mismatch with squeeze
                    target = data.y.long().squeeze()
                    correct += (pred == target).sum().item()
            
            avg_loss = total_loss / len(train_loader)
            acc = correct / len(test_data)
            print(f"Epoch {epoch:03d} | Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}")

        # 5. Save Trained Model Weights
        # Ensure the output directory exists
        os.makedirs(os.path.dirname(self.config.trained_model_path), exist_ok=True)
        
        torch.save(model.state_dict(), self.config.trained_model_path)
        print(f"Trained weights saved to: {self.config.trained_model_path}")

In [ ]:
try:
    config_manager = ConfigurationManager()
    training_config = config_manager.get_training_config()
    trainer = ModelTrainer(training_config)
    trainer.train_model()
except Exception as e:
    print(f"Error during training: {e}")
    